In [8]:
import tensorflow as tf

In [9]:
import argparse
import sys
from pathlib import Path

import tensorflow as tf



In [10]:
DATA_DIR = Path("../data").resolve()

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_crnn_model(
    input_shape=(32, 128, 1), # عرض تصویر معمولاً برای توالی‌ها بزرگ‌تر از ارتفاع است
    num_classes=45            # تعداد کلاس‌ها (شامل کاراکترها + علامت blank برای CTC)
):
    inputs = layers.Input(shape=input_shape, name="image_input")

    # ==========================================
    # 1. CNN Feature Extractor
    # ==========================================
    # Block 1
    x = layers.Conv2D(64, (3, 3), padding="same", name="conv1")(inputs)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn1")(x)
    x = layers.ReLU(name="relu1")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), name="pool1")(x)

    # Block 2
    x = layers.Conv2D(128, (3, 3), padding="same", name="conv2")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn2")(x)
    x = layers.ReLU(name="relu2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), name="pool2")(x)

    # Block 3 & 4
    x = layers.Conv2D(256, (3, 3), padding="same", name="conv3")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn3")(x)
    x = layers.ReLU(name="relu3")(x)

    x = layers.Conv2D(256, (3, 3), padding="same", name="conv4")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn4")(x)
    x = layers.ReLU(name="relu4")(x)
    x = layers.MaxPooling2D(pool_size=(2, 1), strides=(2, 1), name="pool3")(x)

    # Block 5 & 6
    x = layers.Conv2D(512, (3, 3), padding="same", name="conv5")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn5")(x)
    x = layers.ReLU(name="relu5")(x)

    x = layers.Conv2D(512, (3, 3), padding="same", name="conv6")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn6")(x)
    x = layers.ReLU(name="relu6")(x)
    x = layers.MaxPooling2D(pool_size=(2, 1), strides=(2, 1), name="pool4")(x)

    # Block 7
    x = layers.Conv2D(512, (3, 3), padding="same", name="conv7")(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=1e-5, name="bn7")(x)
    x = layers.ReLU(name="relu7")(x)

    # ==========================================
    # 2. Map to Sequence
    # ==========================================
    # ورودی در این مرحله ابعادی معادل (Batch, Height, Width, Channels) دارد.
    # ارتفاع به ۱ کاهش یافته (در ورودی H=32: 32 -> 16 -> 8 -> 4 -> 2 -> 1)
    # خروجی Reshape به صورت (Batch, Width, Height * Channels) در می‌آید.
    
    # تغییر ابعاد: تبدیل ارتفاع و کانال‌ها به یک بُعد ویژگی
    target_shape = (-1, x.shape[2], x.shape[1] * x.shape[3]) # (TimeSteps, Features)
    x = layers.Permute((2, 1, 3))(x) # جا به جایی عرض و ارتفاع جهت همخوانی با گام زمان
    x = layers.Reshape((-1, x.shape[2] * x.shape[3]), name="reshape")(x)

    # معادل لایه Linear(1024, 96) در مدل PyTorch
    x = layers.Dense(96, activation="relu", name="map2seq")(x)

    # ==========================================
    # 3. Recurrent Layers (BiLSTM)
    # ==========================================
    # rnn1: LSTM(96, 256, bidirectional=True)
    x = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True), name="rnn1"
    )(x)

    # rnn2: LSTM(512, 256, bidirectional=True)
    x = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True), name="rnn2"
    )(x)

    # ==========================================
    # 4. Classifier
    # ==========================================
    # خروجی احتمالات برای هر گام زمانی (Time Step)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="CRNN_Image2Text")
    return model

# ساخت مدل
model = build_crnn_model(input_shape=(32, 128, 1), num_classes=45)
model.summary()

Model: "CRNN_Image2Text"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 32, 128, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 32, 128, 64)    │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 32, 128, 64)    │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (ReLU)                    │ (None, 32, 128, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 16, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 16, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 16, 64, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (ReLU)                    │ (None, 16, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 8, 32, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 8, 32, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 8, 32, 256)     │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (ReLU)                    │ (None, 8, 32, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 8, 32, 256)     │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn4 (BatchNormalization)        │ (None, 8, 32, 256)     │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu4 (ReLU)                    │ (None, 8, 32, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 4, 32, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5 (Conv2D)                  │ (None, 4, 32, 512)     │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn5 (BatchNormalization)        │ (None, 4, 32, 512)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu5 (ReLU)                    │ (None, 4, 32, 512)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv6 (Conv2D)                  │ (None, 4, 32, 512)     │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn6 (BatchNormalization)        │ (None, 4, 32, 512)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu6 (ReLU)                    │ (None, 4, 32, 512)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool4 (MaxPooling2D)            │ (None, 2, 32, 512)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv7 (Conv2D)                  │ (None, 2, 32, 512)     │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn7 (BatchNormalization)        │ (None, 2, 32, 512)     │         2,04

 Total params: 9,287,821 (35.43 MB)

 Trainable params: 9,283,341 (35.41 MB)

 Non-trainable params: 4,480 (17.50 KB)

In [12]:
def ctc_loss_lambda(y_true, y_pred):
    # y_pred: (batch_size, time_steps, num_classes)
    # y_true: (batch_size, max_label_len)
    
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int32")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int32")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int32")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int32")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int32")

    return tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)

# کامپایل مدل
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=ctc_loss_lambda)

In [13]:
import tensorflow as tf

def get_dataset(path):

    train_ds = tf.keras.utils.image_dataset_from_directory(
        path,
        validation_split=0.3,
        subset="training",
        seed=225,
        image_size=(32,32),
        batch_size=8,
        color_mode="grayscale"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        path,
        validation_split=0.3,
        subset="validation",
        seed=225,
        image_size=(32,32),
        batch_size=8,
        color_mode="grayscale"
    )

    rescale = tf.keras.layers.Rescaling(1./255)

    train_ds = train_ds.map(
        lambda x, y: (rescale(x), y)
    )

    val_ds = val_ds.map(
        lambda x, y: (rescale(x), y)
    )

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.prefetch(AUTOTUNE)

    return train_ds, val_ds

In [34]:
train_ds, val_ds = get_dataset("C:\\Users\\MEHDI\\Desktop\\Projects\\persian-letters-ocr\\data\\synthetic_plates_crnn")

TypeError: in user code:

    File "C:\Users\MEHDI\AppData\Local\Temp\ipykernel_18804\3427017517.py", line 60, in process_sample  *
        chars = tf.strings.unicode_split(label_text, precision="UTF-8")

    TypeError: Got an unexpected keyword argument 'precision'


In [36]:
import pathlib
import tensorflow as tf

# ==========================================
# ۱. تنظیم کاراکترها و دیکشنری (Vocabulary)
# ==========================================
# لیست کامل کاراکترهای ممکن در پلاک‌ها (اعداد فارسی + حروف)
DIGITS = ["۰", "۱", "۲", "۳", "۴", "۵", "۶", "۷", "۸", "۹"]
LETTERS = [
    "الف", "ب", "پ", "ت", "ث", "ج", "چ", "ح", "خ", "د", "ذ", "ر",
    "ز", "ژ", "س", "ش", "ص", "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ک", "گ", "ل", "م", "ن", "و", "ه", "ی"
]

VOCABULARY = DIGITS + LETTERS
MAX_LABEL_LEN = 10  # طول دنباله (مثلاً ۲ رقم + ۱ حرف + ۳ رقم + ۲ رقم + فاصله)

# ساخت لایه StringLookup برای نگاشت کاراکترها به عدد
char_to_num = tf.keras.layers.StringLookup(
    vocabulary=VOCABULARY, mask_token=None, num_oov_indices=1  # index 0 reserved/OOV
)

# ==========================================
# ۲. تابع خواندن labels.txt
# ==========================================
def load_labels_file(data_dir):
    data_dir = pathlib.Path(data_dir)
    labels_path = data_dir / "labels.txt"
    images_dir = data_dir / "images"

    img_paths = []
    labels = []

    with open(labels_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            img_name = parts[0]
            # حذف فاصله‌های اضافی و یکدست‌سازی لیبل
            label_text = parts[1].replace(" ", "")

            img_paths.append(str(images_dir / img_name))
            labels.append(label_text)

    return img_paths, labels

# ==========================================
# ۳. تابع پیش‌پردازش هر نمونه (Image & Label)
# ==========================================
def process_sample(img_path, label_text):
    # الف) پردازش تصویر
    img = tf.io.read_file(img_path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)  # نرمال‌سازی بین 0.0 تا 1.0
    img = tf.image.resize(img, [32, 160])  # (Height=32, Width=160)

    # ب) پردازش متن و تبدیل به اعداد (Tensor)
    chars = tf.strings.unicode_split(label_text, input_encoding="UTF-8")
    label_encoded = char_to_num(chars)

    # ج) پد کردن (Padding) تا طول ثابت MAX_LABEL_LEN
    pad_len = MAX_LABEL_LEN - tf.shape(label_encoded)[0]
    label_encoded = tf.pad(label_encoded, [[0, pad_len]], constant_values=0)

    return {"image_input": img}, label_encoded

# ==========================================
# ۴. تابع اصلی ساخت tf.data.Dataset
# ==========================================
def get_dataset(data_dir="../data/synthetic_plates_crnn", batch_size=32, validation_split=0.2):
    # ۱. دریافت تمام مسیرها و لیبل‌ها
    img_paths, labels = load_labels_file(data_dir)

    num_samples = len(img_paths)
    val_size = int(num_samples * validation_split)

    # آرایش تصادفی داده‌ها
    indices = tf.range(num_samples)
    indices = tf.random.shuffle(indices, seed=42)

    img_paths = tf.gather(img_paths, indices)
    labels = tf.gather(labels, indices)

    # تقسیم به Train و Validation
    train_paths, val_paths = img_paths[val_size:], img_paths[:val_size]
    train_labels, val_labels = labels[val_size:], labels[:val_size]

    # ساخت دیتاست
    train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = (
        train_ds.map(process_sample, num_parallel_calls=AUTOTUNE)
        .shuffle(buffer_size=5000)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

    val_ds = (
        val_ds.map(process_sample, num_parallel_calls=AUTOTUNE)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

    return train_ds, val_ds

In [26]:

from PIL import Image

In [37]:
# ۱. دریافت دیتاست‌ها
train_ds, val_ds = get_dataset("../data/synthetic_plates_crnn", batch_size=64)

# ۲. ساخت مدل (ابعاد تصویر 32x160 و تعداد کلاس‌ها برابر با طول Vocabulary + 1)
num_classes = len(VOCABULARY) + 2  # تعداد کل کاراکترها + OOV + Blank token در CTC
model = build_crnn_model(input_shape=(32, 160, 1), num_classes=num_classes)

# ۳. کامپایل مدل با CTC Loss
def ctc_loss_lambda(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    return tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=ctc_loss_lambda)

# ۴. اجرا
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25
)

Epoch 1/25

625/625 ━━━━━━━━━━━━━━━━━━━━ 1788s 3s/step - loss: 19.3114 - val_loss: 0.7687
Epoch 2/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 1584s 3s/step - loss: 0.3599 - val_loss: 0.1825
Epoch 3/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 1471s 2s/step - loss: 0.0752 - val_loss: 0.0113
Epoch 4/25
470/625 ━━━━━━━━━━━━━━━━━━━━ 5:40 2s/step - loss: 0.0505

: 